In [1]:
# --- Setup (works locally and in Google Colab) ---
import sys, os
from pathlib import Path
if "google.colab" in sys.modules and not Path("src").exists():
    !git clone https://github.com/ArjunGopal10/privacy-preserving-synthetic-graph-generation.git
    %cd privacy-preserving-synthetic-graph-generation
    !pip install -q -r requirements.txt
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np, pandas as pd, networkx as nx, matplotlib.pyplot as plt
from src.visualization.plot_style import apply_style
apply_style(); plt.rcParams["figure.dpi"] = 100
%matplotlib inline

# 04 · Community metrics
**Protocol A**: score each release under the fixed P0. **Protocol B**: re-detect communities on the release and compare with P0.

In [2]:
from src.graph_generation.graph_loader import load_karate_club
from src.graph_generation.graph_preprocessing import graph_to_pair_vector, pair_vector_to_graph
from src.graph_generation.randomized_response import draw_uniforms, randomized_response_vectors
from src.community.community_detection import detect_communities
from src.community.community_metrics import fixed_partition_report, redetected_report
G = load_karate_club(); P0 = detect_communities(G, "louvain", seed=42); vec = graph_to_pair_vector(G)
U = draw_uniforms(561, 10, 42); rows = []
for e in [0.5, 2, 5, 8]:
    for k, rel in enumerate(randomized_response_vectors(vec, e, U)):
        H = pair_vector_to_graph(rel, 34); r = redetected_report(H, P0, "louvain", seed=42); r.pop("sizes")
        rows.append({"epsilon": e, **fixed_partition_report(H, P0), **r})
df = pd.DataFrame(rows).groupby("epsilon").mean()
df[["modularity_fixed", "coverage_fixed", "conductance_fixed", "modularity_redetected", "nmi", "ari", "n_communities"]].round(3)

,modularity_fixed,coverage_fixed,conductance_fixed,modularity_redetected,nmi,ari,n_communities
epsilon,,,,,,,
0.5,0.018,0.431,0.658,0.136,0.134,0.047,4.1
2.0,0.188,0.608,0.482,0.253,0.357,0.252,4.2
5.0,0.361,0.783,0.293,0.376,0.771,0.692,3.8
8.0,0.385,0.807,0.269,0.385,1.000,1.000,3.0


Note how the **re-detected** modularity stays clearly positive even at small epsilon (modularity optimisers fit noise), whereas modularity under the fixed P0 and NMI vs P0 fall to ≈0.